# Fine-tuning tactile paving satu kelas

Notebook ini hanya menerima export training yang sudah lolos gate lokal. Output tetap berstatus candidate dan bukan izin deployment atau penggunaan berjalan mandiri. Jalankan dengan runtime GPU di Google Colab.

In [ ]:
!pip install -q "ultralytics==8.4.138" pyyaml


In [ ]:
!git lfs install
!git clone --branch codex/model-first-mobile-ready --single-branch https://github.com/Riqqi15/Yoloooo.git /content/Yoloooo
!cp -r /content/Yoloooo/artifacts/datasets/station-tactile-v2 /content/station-v2
!cp /content/Yoloooo/models/guidetwsi/yolo11n_tactile.pt /content/yolo11n_tactile.pt

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys

import torch
import ultralytics
import yaml
from ultralytics import YOLO

assert torch.cuda.is_available(), "GPU Colab wajib tersedia"
assert ultralytics.__version__ == "8.4.138", ultralytics.__version__

DATASET_ROOT = Path("/content/station-v2")
CHECKPOINT = Path("/content/yolo11n_tactile.pt")
RUN_NAME = "tactile-one-class-v2"
RUNS_ROOT = Path("/content/runs/tactile")
CANDIDATE_DIR = Path("/content/artifacts/candidates") / RUN_NAME


In [ ]:
EXPORT_MANIFEST = DATASET_ROOT / "export_manifest.json"
TACTILE_ROOT = DATASET_ROOT / "tactile"
DATA_YAML = TACTILE_ROOT / "data.yaml"
required_files = (EXPORT_MANIFEST, DATA_YAML, CHECKPOINT)
for path in required_files:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"File wajib hilang/kosong: {path}")

split_files = {}
for split in ("train", "val"):
    image_dir = TACTILE_ROOT / "images" / split
    label_dir = TACTILE_ROOT / "labels" / split
    images = sorted(path for path in image_dir.glob("*") if path.is_file())
    labels = sorted(label_dir.glob("*.txt"))
    if not images or not labels:
        raise ValueError(f"Split {split} harus memiliki image dan label")
    split_files[split] = {"images": images, "labels": labels}

data_config = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
raw_names = data_config.get("names", {})
names = {int(index): str(name) for index, name in raw_names.items()} if isinstance(raw_names, dict) else {index: str(name) for index, name in enumerate(raw_names)}
if names != {0: "tactile_paving"}:
    raise ValueError(f"Taxonomy harus satu kelas tactile_paving, ditemukan {names}")

export_manifest = json.loads(EXPORT_MANIFEST.read_text(encoding="utf-8"))
if export_manifest.get("validation", {}).get("training_ready") is not True:
    raise ValueError("Export manifest tidak berstatus training_ready")
dataset_version = export_manifest.get("dataset_version")
if dataset_version != "station-tactile-v2":
    raise ValueError(f"dataset_version harus station-tactile-v2, ditemukan {dataset_version!r}")
print({"dataset_version": dataset_version, "train_images": len(split_files["train"]["images"]), "val_images": len(split_files["val"]["images"])})


In [ ]:
model = YOLO(str(CHECKPOINT), task="segment")
smoke_result = model.predict(source=str(split_files["train"]["images"][0]), imgsz=640, device=0, verbose=False)
if not smoke_result:
    raise RuntimeError("Smoke inference tidak menghasilkan result object")
print("Smoke inference berhasil")


In [ ]:
model.train(
    data=str(DATA_YAML),
    task="segment",
    epochs=80,
    imgsz=640,
    batch=-1,
    device=0,
    seed=42,
    deterministic=True,
    project=str(RUNS_ROOT),
    name=RUN_NAME,
    hsv_h=0.01, hsv_s=0.25, hsv_v=0.20,
    degrees=3.0, translate=0.05, scale=0.20, perspective=0.0002,
    fliplr=0.5, flipud=0.0, mosaic=0.2, mixup=0.0, copy_paste=0.0,
)
BEST_PT = RUNS_ROOT / RUN_NAME / "weights" / "best.pt"
if not BEST_PT.is_file():
    raise FileNotFoundError(f"Checkpoint terbaik tidak ditemukan: {BEST_PT}")


In [ ]:
best_model = YOLO(str(BEST_PT), task="segment")
best_names = {int(index): str(name) for index, name in best_model.names.items()}
if best_model.task != "segment" or best_names != {0: "tactile_paving"}:
    raise ValueError(f"Kontrak checkpoint salah: task={best_model.task}, labels={best_names}")
validation = best_model.val(data=str(DATA_YAML), imgsz=640, device=0, split="val")
metrics = {key: float(value) for key, value in validation.results_dict.items()}

if CANDIDATE_DIR.exists():
    shutil.rmtree(CANDIDATE_DIR)
CANDIDATE_DIR.mkdir(parents=True)
shutil.copy2(BEST_PT, CANDIDATE_DIR / "best.pt")
(CANDIDATE_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
training_config = {
    "dataset_version": dataset_version,
    "seed": 42,
    "imgsz": 640,
    "epochs": 80,
    "ultralytics_version": "8.4.138",
}
(CANDIDATE_DIR / "training_config.json").write_text(json.dumps(training_config, indent=2), encoding="utf-8")
environment = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(CANDIDATE_DIR / "environment.txt").write_text(environment, encoding="utf-8")
model_card = f"""# Tactile candidate {RUN_NAME}

Status: candidate only; not approved for deployment.

Dataset: {dataset_version}
Task: segment
Labels: tactile_paving
Ultralytics: 8.4.138
Seed: 42
Image size: 640
Epochs: 80

This offline model does not validate safe walking.
"""
(CANDIDATE_DIR / "MODEL_CARD.md").write_text(model_card, encoding="utf-8")
checkpoint_hash = hashlib.sha256((CANDIDATE_DIR / "best.pt").read_bytes()).hexdigest()
(CANDIDATE_DIR / "sha256.txt").write_text(f"{checkpoint_hash}  best.pt\n", encoding="utf-8")
archive_path = shutil.make_archive(str(CANDIDATE_DIR), "zip", root_dir=CANDIDATE_DIR.parent, base_dir=CANDIDATE_DIR.name)
from google.colab import files
files.download(archive_path)
print({"candidate_dir": str(CANDIDATE_DIR), "archive": archive_path, "sha256": checkpoint_hash, "metrics": metrics})
